# Fade margins as a function of information: the central tables

Collects the static benchmark (notebooks 02-08) and the history-rung benchmark (15-17) into one table of information rung times model
family, with the hold-out RMSE and the calibrated 99 % margin in each cell; then the unseen-link transfer per family, the margins that follow
the state of the link (quantile models), and the sparse-uplink test.

The static rung is read from the residual files of notebooks 02-08 on all hold-out rows; the history rungs from the files of 15-17 on the
rows with a complete history (99.9 % of the hold-out). Margin rule as in `11_FADE MARGIN ANALYSIS`.

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from history_features import build, RUNGS

RANDOM_STATE = 42
TARGETS = (0.90, 0.95, 0.99)
FAMILIES = ["Linear", "RF", "XGB", "LGBM", "KNN", "ANN", "GRU"]
LABEL = {"Linear": "Linear", "RF": "Random forest", "XGB": "XGBoost", "LGBM": "LightGBM", "KNN": "kNN", "ANN": "ANN", "GRU": "GRU"}
STATIC_FILE = {"Linear": "MLR", "RF": "RF", "XGB": "XGB", "LGBM": "LGBM", "KNN": "KNN", "ANN": "ANN"}      # notebooks 02-08
RUNG_ORDER = ["static", "H1", "H4", "H6", "H7"]
RUNG_LABEL = {"static": "geometry + sensors", "H1": "+ link history", "H4": "+ channel, clock, SF", "H6": "+ earlier SNR, sensors", "H7": "+ link level, geometry"}

def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

def mixture_quantile(w, mu, sd, q):
    lo, hi = float((mu - 10 * sd).min()), float((mu + 10 * sd).max())
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float((w * norm.cdf((mid - mu) / sd)).sum()) < q else (lo, mid)
    return 0.5 * (lo + hi)

def margins(e_cal, e_test):
    g = GaussianMixture(n_components=3, covariance_type="full", random_state=RANDOM_STATE).fit(e_cal.reshape(-1, 1))
    w, mu, sd = g.weights_, g.means_.ravel(), np.sqrt(g.covariances_.reshape(-1))
    out = {}
    for rho in TARGETS:
        emp = float(np.quantile(e_cal, rho))
        fm = max(emp, mixture_quantile(w, mu, sd, rho)) if rho >= 0.98 else emp
        out[f"margin {rho:.2f}"], out[f"reached {rho:.2f}"] = fm, float((e_test <= fm).mean())
    return out


In [ ]:
# Static rung from the residual files of notebooks 02-08, history rungs from 15-17
rows = {}
for fam, f in STATIC_FILE.items():
    oof = pd.read_csv(f"Residuals/residuals_{f}_oof.csv", usecols=["resid_db"])["resid_db"].to_numpy()
    te = pd.read_csv(f"Residuals/residuals_{f}_test.csv", usecols=["resid_db"])["resid_db"].to_numpy()
    rows[(fam, "static")] = {"hold-out RMSE": rmse(te), "hold-out MAE": float(np.abs(te).mean()), "OOF RMSE": rmse(oof), **margins(oof, te)}
static = pd.DataFrame(rows).T
hist = pd.concat([pd.read_csv(f, index_col=[0, 1]) for f in ("CV_Results/history_benchmark.csv", "CV_Results/history_benchmark_KNN.csv", "CV_Results/history_benchmark_GRU.csv")])
references = hist.loc[hist.index.get_level_values(1) == "-"]
hist = hist.loc[hist.index.get_level_values(1) != "-"]
table = pd.concat([static, hist])
table.index.names = ["family", "rung"]
table.to_csv("CV_Results/information_by_family.csv")

def grid(col, digits):
    g = table[col].unstack("rung").reindex(index=FAMILIES, columns=RUNG_ORDER)
    g.columns = [RUNG_LABEL[c] for c in g.columns]
    return g.round(digits)

print("references (no fitted model): hold-out RMSE and 99 % margin")
print(references[["hold-out RMSE", "margin 0.99", "reached 0.99"]].round(3).to_string())
print("\nhold-out RMSE (dB)")
print(grid("hold-out RMSE", 3).to_string())
print("\nfade margin for 99 % reliability (dB), calibrated on the validation folds")
print(grid("margin 0.99", 2).to_string())
print("\nreliability reached on the hold-out with that margin")
print(grid("reached 0.99", 4).to_string())


In [ ]:
# Figure: the two quantities against the information rung, one line per family
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(RUNG_ORDER))
for fam in FAMILIES:
    for ax, col in ((axA, "hold-out RMSE"), (axB, "margin 0.99")):
        v = [table[col].get((fam, r), np.nan) for r in RUNG_ORDER]
        ax.plot(x, v, marker="o", linewidth=2, markersize=6, markeredgecolor="black", label=LABEL[fam])
for ax, col, ylabel in ((axA, "hold-out RMSE", "Hold-out RMSE (dB)"), (axB, "margin 0.99", "Fade margin for 99 % reliability (dB)")):
    for name, ls in (("link mean", "--"), ("previous-hour mean", ":")):
        ax.axhline(references.loc[(name, "-"), col], linestyle=ls, color="0.4", linewidth=1.2, label=name)
    ax.set_xticks(x)
    ax.set_xticklabels([RUNG_LABEL[r] for r in RUNG_ORDER], rotation=20, ha="right")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, linewidth=0.5, alpha=0.4)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
axA.set_title("(a) what the predictor knows sets the error", loc="left", fontsize=12)
axB.set_title("(b) and the margin", loc="left", fontsize=12)
axB.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
fig.tight_layout()
plt.savefig("Figures/information_by_family.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# A link the model has never seen (rung H4), every family
unseen = pd.concat([pd.read_csv(f, index_col=[0, 1]) for f in ("CV_Results/history_unseen_link.csv", "CV_Results/history_unseen_link_KNN.csv", "CV_Results/history_unseen_link_GRU.csv")])
unseen.index.names = ["family", "link"]
unseen.to_csv("CV_Results/unseen_link_by_family.csv")
summary = unseen.groupby("family").agg(**{"RMSE on the unseen link, mean over links": ("RMSE on the unseen link", "mean"),
                                           "worst link": ("RMSE on the unseen link", "max"), "reached 0.99, mean over links": ("reached on the unseen link", "mean")}).reindex(FAMILIES)
summary["hold-out RMSE with the link in training (H4)"] = [table["hold-out RMSE"].get((f, "H4"), np.nan) for f in FAMILIES]
print(summary.round(3).to_string())
print("\nper link, RMSE on the unseen link (dB)")
print(unseen["RMSE on the unseen link"].unstack("link").reindex(FAMILIES).round(2).to_string())


In [ ]:
# Margins that follow the state of the link: quantile models on rung H4 (and H6 at 99 %)
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values("t").reset_index(drop=True)
fold = d["fold"].to_numpy()
H = {k.split()[0]: v for k, v in RUNGS.items()}
ok = (d["y"].notna() & d[H["H7"]].notna().all(axis=1)).to_numpy()
cal, ev = ok & (fold >= 0) & (fold < 9), ok & (fold == 9)
y, work = d["y"].to_numpy(), d["work"].to_numpy() == 1
X_all = d[H["H7"]].copy()
X_all = X_all.fillna(X_all[ok & (fold < 9)].median())
best = json.load(open("CV_Results/history_best_params.json"))

def oof_and_test(make, cols):
    X = X_all[cols].to_numpy()
    q = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = ok & (fold < k), ok & (fold == k)
        q[va] = make().fit(X[tr], y[tr]).predict(X[va])
    tr = ok & (fold < 9)
    q[ev] = make().fit(X[tr], y[tr]).predict(X[ev])
    return q

def state_margin(q, rho):
    """The quantile prediction shifted by a constant chosen on the validation folds, so that the margin is calibrated like the fixed one."""
    m = q + float(np.quantile((y - q)[cal], rho))
    return {"mean margin": float(m[ev].mean()), "in working hours": float(m[ev & work].mean()), "other hours": float(m[ev & ~work].mean()), "reached": float((y[ev] <= m[ev]).mean())}

qrows = {}
for rho in TARGETS:
    fm = float(np.quantile(y[cal], rho))
    qrows[(f"{rho:.2f}", "one fixed margin above the previous-hour mean", "H1")] = {"mean margin": fm, "in working hours": fm, "other hours": fm, "reached": float((y[ev] <= fm).mean())}
quantile_models = {
    "LightGBM": lambda rho: lgb.LGBMRegressor(objective="quantile", alpha=rho, subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **best["LGBM"]),
    "XGBoost": lambda rho: xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=rho, device="cuda", tree_method="hist", random_state=RANDOM_STATE, **best["XGB"]),
}
for name, make in quantile_models.items():
    for rung, rhos in (("H4", TARGETS), ("H6", (0.99,))):
        for rho in rhos:
            t0 = time.time()
            qrows[(f"{rho:.2f}", f"{name} quantile model", rung)] = state_margin(oof_and_test(lambda: make(rho), H[rung]), rho)
            print(f"{name} {rung} rho={rho}: {qrows[(f'{rho:.2f}', f'{name} quantile model', rung)]['mean margin']:.2f} dB mean margin ({time.time() - t0:.0f} s)")


In [ ]:
# ANN with the pinball loss at 99 %, rungs H4 and H6 (same network and protocol as notebook 15)
import tensorflow as tf
from tensorflow import keras
tf.config.set_visible_devices([], "GPU")

class ANNQuantile:
    def __init__(self, layers, rho, seeds=(0, 1, 2)):
        self.layers, self.rho, self.seeds, self.scaler, self.nets = layers, rho, seeds, StandardScaler(), []
    def fit(self, X, y):
        Xs = self.scaler.fit_transform(X).astype(np.float32); y = y.astype(np.float32)
        n_val = max(1, len(Xs) // 10)
        rho = self.rho
        pinball = lambda t, q: tf.reduce_mean(tf.maximum(rho * (t - q), (rho - 1) * (t - q)))
        self.nets = []
        for seed in self.seeds:
            tf.keras.utils.set_random_seed(seed)
            net = keras.Sequential([keras.Input(shape=(Xs.shape[1],))] + [keras.layers.Dense(n, activation="relu") for n in self.layers] + [keras.layers.Dense(1)])
            net.compile(optimizer=keras.optimizers.Adam(1e-3), loss=pinball)
            net.fit(Xs[:-n_val], y[:-n_val], validation_data=(Xs[-n_val:], y[-n_val:]), epochs=200, batch_size=2048, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True), keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5)])
            self.nets.append(net)
        return self
    def predict(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        return np.mean([net.predict(Xs, batch_size=8192, verbose=0).ravel() for net in self.nets], axis=0)

layers = tuple(best["ANN"])
for rung in ("H4", "H6"):
    t0 = time.time()
    qrows[("0.99", "ANN quantile model", rung)] = state_margin(oof_and_test(lambda: ANNQuantile(layers, 0.99), H[rung]), 0.99)
    print(f"ANN {rung} rho=0.99: {qrows[('0.99', 'ANN quantile model', rung)]['mean margin']:.2f} dB mean margin ({time.time() - t0:.0f} s)")

state = pd.DataFrame(qrows).T
state.index.names = ["target", "model", "rung"]
state.to_csv("CV_Results/state_margins.csv")
print(state.round(3).to_string())


In [ ]:
# Sparser uplinks: every 10th and every 60th packet of each link kept, anchor = mean of the previous 20 kept packets, LightGBM on rung H4.
# The references are compared on the rows where all of them exist.
srows = {}
raw = pd.concat([pd.read_csv("Data_Files/train.csv").assign(fold=np.load("Data_Files/train_folds.npy")), pd.read_csv("Data_Files/test.csv").assign(fold=9)], ignore_index=True)
raw["t"] = pd.to_datetime(raw["time"], utc=True, format="mixed")
raw = raw.drop(columns="time").sort_values(["device_id", "t"])
for k in (10, 60):
    s = build(raw[raw.groupby("device_id").cumcount() % k == 0], anchor="m20")
    s_fold, s_ok = s["fold"].to_numpy(), s["y"].notna().to_numpy()      # windows shorter than the uplink interval are undefined; LightGBM takes them as missing
    s_cal, s_ev = s_ok & (s_fold >= 0) & (s_fold < 9), s_ok & (s_fold == 9)
    Xs, ys = s[H["H4"]].to_numpy(), s["y"].to_numpy()
    q = np.full(len(s), np.nan)
    for f in range(5):
        tr, va = s_ok & (s_fold < f), s_ok & (s_fold == f)
        q[va] = lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **best["LGBM"]).fit(Xs[tr], ys[tr]).predict(Xs[va])
    tr = s_ok & (s_fold < 9)
    q[s_ev] = lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **best["LGBM"]).fit(Xs[tr], ys[tr]).predict(Xs[s_ev])
    errors = {"previous kept packet": ys - s["r_lag1"].to_numpy(), "mean of the previous 20 kept packets": ys, "mean of the previous 24 hours": ys - s["r_m24h"].to_numpy(), "LightGBM on rung H4": ys - q}
    have = np.all([~np.isnan(e) for e in errors.values()], axis=0)
    for name, e in errors.items():
        fm = float(np.quantile(e[s_cal & have], 0.99))
        srows[(f"every {k} min", name)] = {"hold-out RMSE": rmse(e[s_ev & have]), "margin 0.99": fm, "reached": float((e[s_ev & have] <= fm).mean()), "hold-out rows": int((s_ev & have).sum())}
sparse = pd.DataFrame(srows).T
sparse.index.names = ["uplink interval", "predictor"]
sparse.to_csv("CV_Results/sparse_uplinks.csv")
print(sparse.round(3).to_string())
